# Part C: cost per user action and break-even

Streaming-Order AMM, FRE-GY 6931. Questions:

1. What does each user action cost in USD, from **measured** gas and a **stated** gas price and ETH price?
2. After the keeper's gas, when does slicing still beat a lump sum?
3. What monthly volume and number of orders cover running costs, given a swap-fee share and an execution fee?

Gas sources: `docs/data/gas_costs_local.json` (written by `packages/hardhat/scripts/measureCostGas.ts`, exact `gasUsed`
from receipts) for the order lifecycle, deployments, approvals and the arbitrage swap; `docs/gas-pool.md` for liquidity
actions. All gas is from a local Hardhat chain; Sepolia/mainnet execution is the same EVM, but numbers can move by a few
percent with account state (for example the first token transfer to a fresh address).

**Important:** the contracts as built charge **no execution fee** and send **all** of the 30 bps swap fee to LPs. The revenue
side of question 3 is therefore the business model *proposed* in the project proposal, not something the deployed contracts
do. Section 4 shows the as-built case (zero protocol revenue) explicitly.

In [1]:
import json, math
from pathlib import Path
import pandas as pd

gas = json.loads(Path("data/gas_costs_local.json").read_text())["gasUsed"]

# From docs/gas-pool.md (after the packing change), same method, not re-measured here:
gas_pool_md = {
    "addLiquidity_existing_lp": 123_548,
    "addLiquidity_new_lp": 157_772,
    "removeLiquidity_half": 111_431,
}
pd.Series({**gas, **gas_pool_md}, name="gasUsed").to_frame()

,gasUsed
deploy_MockWBTC,636703
deploy_MockUSDC,637303
deploy_Pool,1749662
deploy_PriceGuard,608992
deploy_OrderBook,2268674
faucet_wbtc_first,50856
approve_wbtc_exact,46353
createOrder_first,158734
executeSlice_first,190399
executeSlice_middle,156199


## 1. Assumptions (all in one place)

| Symbol | Value | Why |
|---|---|---|
| `GAS_GWEI_BASE` | 10 gwei | A plausible Ethereum L1 price on a normal day. Not observed; the sensitivity grid covers 1 to 50 gwei. |
| `ETH_USD` | 3,000 | Round assumption for the ETH price. Grid covers 2,000 to 5,000. |
| `L2_GAS_GWEI` | 0.05 gwei | An L2-like execution price, to show how much of the cost is the chain, not the design. L1 data fees on rollups are ignored. |
| `BTC_USD` | 60,000 | Same reference price as the tests, the notebooks and the mock feed. |

Gas price × gas used × ETH price = USD. Nothing here is a forecast.

In [2]:
GAS_GWEI_BASE = 10
ETH_USD = 3_000
L2_GAS_GWEI = 0.05
BTC_USD = 60_000

def usd(gas_used, gwei=GAS_GWEI_BASE, eth_usd=ETH_USD):
    return gas_used * gwei * 1e-9 * eth_usd

assert abs(usd(100_000) - 3.0) < 1e-9   # 100k gas at 10 gwei and $3,000 is $3.00

## 2. USD cost of each user action

In [3]:
actions = [
    # (who pays, action, gas)
    ("Seller", "Approve WBTC to the OrderBook (exact amount)", gas["approve_wbtc_exact"]),
    ("Seller", "createOrder", gas["createOrder_first"]),
    ("Seller", "claim", gas["claim_first"]),
    ("Seller", "cancel (unfilled order)", gas["cancel_unfilled"]),
    ("Seller", "cancel (finished order)", gas["cancel_finished"]),
    ("Keeper", "executeSlice (mean of 4 slices)", gas["executeSlice_mean_4"]),
    ("Keeper", "executeSlice (first slice, worst case)", gas["executeSlice_first"]),
    ("Trader / arb", "swapExactIn", gas["swapExactIn_usdc_in"]),
    ("Trader / arb", "Approve token to the Pool", gas["approve_usdc_exact"]),
    ("LP", "addLiquidity (existing LP)", gas_pool_md["addLiquidity_existing_lp"]),
    ("LP", "addLiquidity (new LP)", gas_pool_md["addLiquidity_new_lp"]),
    ("LP", "removeLiquidity", gas_pool_md["removeLiquidity_half"]),
    ("Test only", "Faucet (mock token)", gas["faucet_wbtc_first"]),
]
t = pd.DataFrame(actions, columns=["payer", "action", "gas"])
t["USD @ 10 gwei (L1)"] = t["gas"].map(lambda g: round(usd(g), 2))
t["USD @ 50 gwei (busy L1)"] = t["gas"].map(lambda g: round(usd(g, 50), 2))
t["USD @ 0.05 gwei (L2-like)"] = t["gas"].map(lambda g: round(usd(g, L2_GAS_GWEI), 4))
t

,payer,action,gas,USD @ 10 gwei (L1),USD @ 50 gwei (busy L1),USD @ 0.05 gwei (L2-like)
0,Seller,Approve WBTC to the OrderBook (exact amount),46353,1.39,6.95,0.0070
1,Seller,createOrder,158734,4.76,23.81,0.0238
2,Seller,claim,56705,1.70,8.51,0.0085
3,Seller,cancel (unfilled order),56148,1.68,8.42,0.0084
4,Seller,cancel (finished order),41340,1.24,6.20,0.0062
5,Keeper,executeSlice (mean of 4 slices),163545,4.91,24.53,0.0245
6,Keeper,"executeSlice (first slice, worst case)",190399,5.71,28.56,0.0286
7,Trader / arb,swapExactIn,117775,3.53,17.67,0.0177
8,Trader / arb,Approve token to the Pool,46365,1.39,6.95,0.0070
9,LP,addLiquidity (existing LP),123548,3.71,18.53,0.0185


In [4]:
deploy_total = sum(v for k, v in gas.items() if k.startswith("deploy_") and "Mock" not in k)
deploy_with_mocks = sum(v for k, v in gas.items() if k.startswith("deploy_"))
print(f"Deploy Pool + PriceGuard + OrderBook: {deploy_total:,} gas = ${usd(deploy_total):,.2f} at 10 gwei (one-off)")
print(f"Including the two mock tokens (testnet only): {deploy_with_mocks:,} gas")

Deploy Pool + PriceGuard + OrderBook: 4,627,328 gas = $138.82 at 10 gwei (one-off)
Including the two mock tokens (testnet only): 5,901,334 gas


### Sensitivity: the full seller journey

One order with `k` slices = approve + createOrder + `k` × executeSlice (keeper) + claim. The seller pays the first two and the
last; the keeper pays the slices. The grid shows the **whole** order's gas in USD.

In [5]:
def order_gas(k):
    return gas["approve_wbtc_exact"] + gas["createOrder_first"] + k * gas["executeSlice_mean_4"] + gas["claim_first"]

grid = pd.DataFrame(
    {f"ETH ${e:,}": {f"{g} gwei": round(usd(order_gas(10), g, e), 2) for g in [1, 5, 10, 20, 50]} for e in [2_000, 3_000, 5_000]}
)
print(f"10-slice order, {order_gas(10):,} gas in total. USD:")
grid

10-slice order, 1,897,242 gas in total. USD:


,"ETH $2,000","ETH $3,000","ETH $5,000"
1 gwei,3.79,5.69,9.49
5 gwei,18.97,28.46,47.43
10 gwei,37.94,56.92,94.86
20 gwei,75.89,113.83,189.72
50 gwei,189.72,284.59,474.31


## 3. Does slicing still pay after the keeper's gas?

Same integer constant-product model as `Pool.sol` and `analysis_slicing.ipynb` (30 bps fee from the input). Pool:
500 WBTC / 30,000,000 USDC. "Sliced" assumes the pool is **fully** refilled before each slice (the best case; Part B shows how
the saving shrinks with partial refill). Saving = sliced proceeds − lump-sum proceeds − keeper gas for all slices.

In [6]:
R0, R1 = 500 * 10**8, 30_000_000 * 10**6   # raw WBTC (8 dec), raw USDC (6 dec)

def amount_out(a, r_in, r_out, fee_bps=30):
    w = a * (10_000 - fee_bps)
    return w * r_out // (r_in * 10_000 + w)

def slice_amounts(total, k):
    s = total // k
    return [s] * (k - 1) + [total - s * (k - 1)]

assert amount_out(20 * 10**8, R0, R1) == 1_150_517_367_388   # equals the contract result in test/EndToEnd.ts

rows = []
for btc in [1, 5, 20, 50]:
    total = btc * 10**8
    lump = amount_out(total, R0, R1) / 1e6
    for k in [5, 10, 20]:
        refilled = sum(amount_out(a, R0, R1) for a in slice_amounts(total, k)) / 1e6
        keeper = k * usd(gas["executeSlice_mean_4"])
        rows.append({
            "order (WBTC)": btc, "order (USD)": btc * BTC_USD, "slices": k,
            "saving before gas ($)": round(refilled - lump, 2),
            "keeper gas @10 gwei ($)": round(keeper, 2),
            "net saving ($)": round(refilled - lump - keeper, 2),
            "net saving (bps of order)": round((refilled - lump - keeper) / (btc * BTC_USD) * 1e4, 1),
        })
savings = pd.DataFrame(rows)
savings

,order (WBTC),order (USD),slices,saving before gas ($),keeper gas @10 gwei ($),net saving ($),net saving (bps of order)
0,1,60000,5,95.20,24.53,70.67,11.8
1,1,60000,10,107.12,49.06,58.05,9.7
2,1,60000,20,113.08,98.13,14.95,2.5
3,5,300000,5,2357.37,24.53,2332.84,77.8
4,5,300000,10,2654.68,49.06,2605.62,86.9
5,5,300000,20,2803.56,98.13,2705.44,90.2
6,20,1200000,5,36415.65,24.53,36391.12,303.3
7,20,1200000,10,41130.34,49.06,41081.28,342.3
8,20,1200000,20,43501.76,98.13,43403.63,361.7
9,50,3000000,5,212692.76,24.53,212668.23,708.9


In [7]:
# Smallest order (in WBTC, 0.1 steps) for which 10 slices beat a lump sum after keeper gas, at each gas price.
def breakeven_order_btc(k=10, gwei=GAS_GWEI_BASE):
    keeper = k * usd(gas["executeSlice_mean_4"], gwei)
    for tenths in range(1, 2001):
        total = tenths * 10**7
        gain = (sum(amount_out(a, R0, R1) for a in slice_amounts(total, k)) - amount_out(total, R0, R1)) / 1e6
        if gain > keeper:
            return tenths / 10
    return None

min_order = {f"{g} gwei": breakeven_order_btc(10, g) for g in [L2_GAS_GWEI, 1, 10, 50]}
min_order

{'0.05 gwei': 0.1, '1 gwei': 0.3, '10 gwei': 0.7, '50 gwei': 1.6}

Reading: in a 500 WBTC pool, slicing a small order is not worth the keeper's gas on L1, because price impact grows with the
square of the order's share of the pool while the keeper cost grows with the number of slices. The minimum worthwhile order
is printed above for each gas price. On an L2-like chain the threshold almost disappears.

## 4. Break-even for going to market

**Revenue (proposed, not built):**
- `PROTOCOL_SHARE_BPS`: protocol's cut of the 30 bps swap fee (Uniswap v2 style "fee switch"). Assumed 5 bps, so LPs keep 25.
- `EXEC_FEE_BPS`: execution fee on each streaming order's notional. Assumed 10 bps. For comparison, the base-case net saving
  of a 20 WBTC, 10-slice order (table above) is far larger, so the fee leaves the seller well ahead.
- `ARB_VOLUME_MULT`: pool volume from arbitrage per unit of streaming volume. Every sliced WBTC is bought back by an
  arbitrageur in the refill, so about 1.0 (the end-to-end test needed one arbitrage trade per slice). The protocol share is
  earned on streaming volume **and** arbitrage volume.

**Costs:**
- Variable: keeper gas, `k` slices per order (paid by us if we run the keeper).
- Fixed per month (assumptions, rounded): RPC provider 50, keeper + bot server 20, front-end hosting 20, monitoring 25,
  audit amortised over 24 months (40,000 / 24 ≈ 1,667). Team salaries are left out; a second scenario adds one
  part-time engineer at 5,000 per month.

Break-even orders per month `N` solve  `N · (S · (EXEC + PROTOCOL · (1 + ARB)) / 10⁴ − k · keeper_cost) = F`.

In [8]:
PROTOCOL_SHARE_BPS = 5
EXEC_FEE_BPS = 10
ARB_VOLUME_MULT = 1.0
FIXED = {"RPC provider": 50, "Keeper and arb server": 20, "Front-end hosting": 20, "Monitoring": 25,
         "Audit amortised (40k / 24 months)": 40_000 / 24}
F_BASE = sum(FIXED.values())
F_WITH_ENGINEER = F_BASE + 5_000

def breakeven(order_usd, k, gwei=GAS_GWEI_BASE, fixed=F_BASE, exec_bps=EXEC_FEE_BPS, protocol_bps=PROTOCOL_SHARE_BPS):
    revenue_per_order = order_usd * (exec_bps + protocol_bps * (1 + ARB_VOLUME_MULT)) / 1e4
    margin = revenue_per_order - k * usd(gas["executeSlice_mean_4"], gwei)
    if margin <= 0:
        return None, None, revenue_per_order, margin
    n = math.ceil(fixed / margin)
    return n, n * order_usd, revenue_per_order, margin

print(f"Fixed costs per month: ${F_BASE:,.0f} (without salaries), ${F_WITH_ENGINEER:,.0f} with one part-time engineer")
pd.Series(FIXED).round(0).to_frame("USD / month")

Fixed costs per month: $1,782 (without salaries), $6,782 with one part-time engineer


,USD / month
RPC provider,50.0
Keeper and arb server,20.0
Front-end hosting,20.0
Monitoring,25.0
Audit amortised (40k / 24 months),1667.0


In [9]:
rows = []
for order_usd in [50_000, 250_000, 1_000_000]:
    for k in [10, 20]:
        for gwei, chain in [(GAS_GWEI_BASE, "L1 10 gwei"), (50, "L1 50 gwei"), (L2_GAS_GWEI, "L2-like")]:
            for fixed, label in [(F_BASE, "no salary"), (F_WITH_ENGINEER, "+1 part-time")]:
                n, vol, rev, margin = breakeven(order_usd, k, gwei, fixed)
                rows.append({"avg order ($)": order_usd, "slices": k, "chain": chain, "costs": label,
                             "revenue / order ($)": round(rev, 2), "margin / order ($)": round(margin, 2),
                             "orders / month": n if n else "never", "streaming volume / month ($)": f"{vol:,.0f}" if vol else "never"})
be = pd.DataFrame(rows)
be[be["costs"] == "no salary"].drop(columns="costs")

,avg order ($),slices,chain,revenue / order ($),margin / order ($),orders / month,streaming volume / month ($)
0,50000,10,L1 10 gwei,100.0,50.94,35,"1,750,000"
2,50000,10,L1 50 gwei,100.0,-145.32,never,never
4,50000,10,L2-like,100.0,99.75,18,"900,000"
6,50000,20,L1 10 gwei,100.0,1.87,952,"47,600,000"
8,50000,20,L1 50 gwei,100.0,-390.64,never,never
10,50000,20,L2-like,100.0,99.51,18,"900,000"
12,250000,10,L1 10 gwei,500.0,450.94,4,"1,000,000"
14,250000,10,L1 50 gwei,500.0,254.68,7,"1,750,000"
16,250000,10,L2-like,500.0,499.75,4,"1,000,000"
18,250000,20,L1 10 gwei,500.0,401.87,5,"1,250,000"


In [10]:
be[be["costs"] == "+1 part-time"].drop(columns="costs")

,avg order ($),slices,chain,revenue / order ($),margin / order ($),orders / month,streaming volume / month ($)
1,50000,10,L1 10 gwei,100.0,50.94,134,"6,700,000"
3,50000,10,L1 50 gwei,100.0,-145.32,never,never
5,50000,10,L2-like,100.0,99.75,68,"3,400,000"
7,50000,20,L1 10 gwei,100.0,1.87,3621,"181,050,000"
9,50000,20,L1 50 gwei,100.0,-390.64,never,never
11,50000,20,L2-like,100.0,99.51,69,"3,450,000"
13,250000,10,L1 10 gwei,500.0,450.94,16,"4,000,000"
15,250000,10,L1 50 gwei,500.0,254.68,27,"6,750,000"
17,250000,10,L2-like,500.0,499.75,14,"3,500,000"
19,250000,20,L1 10 gwei,500.0,401.87,17,"4,250,000"


In [11]:
# Base case used in the report: $250k average order, 10 slices, L1 at 10 gwei, no salary.
n, vol, rev, margin = breakeven(250_000, 10)
print(f"Base case: {n} orders/month, ${vol:,.0f} streaming volume/month, revenue ${rev:,.2f} and margin ${margin:,.2f} per order")
assert n * margin >= F_BASE and (n - 1) * margin < F_BASE   # n is the smallest whole number of orders that covers costs

# As built (no execution fee, no protocol share): revenue is zero and the keeper's gas is a pure cost.
n0, *_ = breakeven(250_000, 10, exec_bps=0, protocol_bps=0)
assert n0 is None
print("As built (no execution fee, all swap fees to LPs): never breaks even; someone must pay the keeper's gas.")

Base case: 4 orders/month, $1,000,000 streaming volume/month, revenue $500.00 and margin $450.94 per order
As built (no execution fee, all swap fees to LPs): never breaks even; someone must pay the keeper's gas.


In [12]:
# Which lever matters: break-even orders per month for the base case while varying one fee at a time.
lever = pd.DataFrame({
    "exec fee (bps)": [0, 5, 10, 20, 10, 10, 10],
    "protocol share (bps)": [5, 5, 5, 5, 0, 2, 10],
})
lever["orders / month"] = [breakeven(250_000, 10, exec_bps=e, protocol_bps=p)[0] or "never" for e, p in zip(lever["exec fee (bps)"], lever["protocol share (bps)"])]
lever

,exec fee (bps),protocol share (bps),orders / month
0,0,5,9
1,5,5,6
2,10,5,4
3,20,5,3
4,10,0,9
5,10,2,6
6,10,10,3


## 5. What this leaves out

- **Gas and ETH prices are assumptions**, not observations; the grids show the range. L2 data-availability fees are ignored.
- **Order flow is assumed**, not estimated from any market data. The break-even counts say how many orders are *needed*, not
  how many we would get.
- **Arbitrage volume = streaming volume** is an approximation from our own test (one refill trade per slice). Real arbitrage
  could be larger (more traders) or smaller (thin interest in a new pool).
- **Full refill** is the best case for slicing; with partial refill the saving shrinks (Part B), which lowers what sellers will
  pay as an execution fee.
- **The fees are not implemented.** Charging them needs contract changes (a protocol fee in `Pool`, a fee in `OrderBook`)
  that the frozen interfaces and the deadline did not allow. The deployed contracts earn the protocol nothing.
- **No salaries, legal or marketing costs** in the base case. One part-time engineer roughly triples the break-even count.
- Gas was measured on a local chain; first-time transfers to fresh addresses cost more than the steady state shown here.